-----
### Tweets Violencia Obstétrica
-----

In [ ]:
!pip install pysentimiento

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.7 MB/s eta 0:00:00


In [ ]:
import torch
import wandb
import os
import pandas as pd
import numpy as np


from datasets import Dataset, Features, ClassLabel, Value, DatasetDict
from datasets import load_dataset
from numpy.ma.extras import average
from sklearn.model_selection import train_test_split

In [ ]:
vo_data = pd.read_excel('/content/GoldStandardVO.xlsx')

In [ ]:
vo_data = vo_data[["Text", "tweet"]]
vo_data = vo_data.rename(columns={"tweet": "label"})

In [ ]:
vo_data['label'].unique()

array(['No violencia obstétrica',
       'Activismo para visibilizar violencia obstétrica',
       'Violencia Obstétrica'], dtype=object)

In [ ]:
mapeo_etiquetas = {'No violencia obstétrica': 1,
                   'Activismo para visibilizar violencia obstétrica': 0,
                   'Violencia Obstétrica': 2
                   }

In [ ]:
data = pd.DataFrame()
data['label'] = vo_data.label.map(mapeo_etiquetas)
data['Text'] = vo_data.Text
data.reset_index(inplace=True, drop=True)

In [ ]:
data


,label,Text
0,1,@DraDarkNTwisty Priva Priva ya te dije cesárea...
1,1,Último fin de semana antes de parir a mi Tesis...
2,1,@luzdlourdes @lara_ortuno @balles72 Se llama e...
3,1,"Korno, ahora que te tengas que ausentar porque..."
4,1,@anapaulacuervo @emmaaax55 Discriminación por ...
...,...,...
2152,2,Llega una puerpera a urgencias a los 5 días de...
2153,2,En mi segundo parto me llevé una inducción y d...
2154,2,Abro hilo de las peores cirugías en las que ha...
2155,2,"MIP en Toco, entré a instrumentar una cesárea ..."


In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
from pysentimiento.preprocessing import preprocess_tweet
from nltk.corpus import stopwords
import string
import re

In [ ]:
swords = set(stopwords.words('spanish'))
for word in ['emoji', 'usuario', 'url', 'link', 'hashtag']:
    swords.add(word)
exclude = set(string.punctuation)

In [ ]:
cleanedData = []
for text in data["Text"]:

    #Limpieza con pysentimiento
    text = preprocess_tweet(text)

    # Remove punctuation
    text = ''.join(ch for ch in text if ch not in exclude)

    # Removing stopwords
    text = [word for word in text.split() if word not in swords]

    # Joining
    text = " ".join(text)

    cleanedData.append(text)

In [ ]:
for i in range(0,5):
    print(cleanedData[i],end="\n\n")

Priva Priva dije cesárea programada 37SDG dejaré papá grabe

Último fin semana parir Tesis hoy momento

Se llama edadismo discriminación motivos edad Se criminaliza edad pecado imperdonable importa si hecho vida productiva si sigues haciendo viejo convierte ciudadano segundaAlgunos creen jóvenes eternamente

Korno ahora ausentar Fer parir dejale changarro buen Zamhashtag zamtástica oscuridad

Discriminación edad Los viejos valemos tres tiras



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif

# https://developers.google.com/machine-learning/guides/text-classification
# Vectorization parameters
# Range (inclusive) of n-gram sizes for tokenizing text.

NGRAM_RANGE = (1, 2) #Unigrams and bigrams

# Limit on the number of features. Top 20K features are used.
TOP_K = 20000

# Whether text should be split into word or character n-grams.
TOKEN_MODE = 'word'

# Document/corpus frequency below which a token will be discarded.
MIN_DOCUMENT_FREQUENCY = 3


# Arguments for tf-idf vectorizer.
kwargs = {
        'ngram_range': NGRAM_RANGE,  # Use Unigrams and bigrams
        'dtype': 'int32',
        'strip_accents': 'unicode',
        'decode_error': 'replace',
        'analyzer': TOKEN_MODE,  # Split text into word tokens.
        'min_df': MIN_DOCUMENT_FREQUENCY,
}
vectorizer = TfidfVectorizer(**kwargs)

In [ ]:
# Learn vocabulary from training texts and vectorize training texts.
BOW = vectorizer.fit_transform(cleanedData)


# Select top 'k' of the vectorized features.
selector = SelectKBest(f_classif, k=min(TOP_K, BOW.shape[1]))
selector.fit(BOW, data['label'])
BOW = selector.transform(BOW).astype('float32')

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. int32 'dtype' will be converted to np.float64.
  warnings.warn(


In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(BOW,np.asarray(data["label"]),
                                                 test_size=0.33, # Proportion of data for the test set
                                                 random_state=42, # for reproducibility
                                                 stratify= np.asarray(data["label"]) )

print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

# Support Vector Machine Classifier Modeling
Everything is ready, now ve can fit our classifier.

In [ ]:
import time

In [ ]:
from sklearn.svm import SVC
start_time = time.time()

model = SVC(kernel='linear')
model.fit(x_train,y_train)

end_time = time.time()
process_time = round(end_time-start_time,2)
print("Fitting SVC took {} seconds".format(process_time))

In [ ]:
predictions = model.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix

print("Accuracy of model is {}%".format(accuracy_score(y_test,predictions) * 100))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
cf_matrix = confusion_matrix(y_test,predictions)
plt.figure(figsize=(5,5))
sns.heatmap(cf_matrix, annot=True, cmap='Blues')

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(y_test, predictions)
print("Classification Report:\n", report)

Using Grid Search for Hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'C': [0.1, 1, 10, 100, 1000],
			'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
			'kernel': ['sigmoid', 'rbf', 'poly', 'linear']}

grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3)

grid.fit(x_train, y_train)


In [ ]:
print(grid.best_params_)

print(grid.best_estimator_)

In [ ]:
grid_predictions = grid.predict(x_test)

In [ ]:
cf_matrix = confusion_matrix(y_test,grid_predictions)
plt.figure(figsize=(5,5))
sns.heatmap(cf_matrix, annot=True, cmap='Blues')

In [ ]:
print("Classification Report:\n", classification_report(y_test, grid_predictions))


# Support Vector Machine Classifier with Cross-validation

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, List, Sequence, Tuple, Union, Optional

import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

ArrayLike = Union[Sequence[int], Sequence[str], np.ndarray]

@dataclass
class CVResult:
    accuracy: float
    f1_macro: float
    f1_weighted: float
    precision_macro: float
    recall_macro: float
    support: int
    confusion: np.ndarray


In [ ]:
def build_pipeline(
    *,
    ngram_range: Tuple[int, int] = (1, 2),
    min_df: int = 2,
    max_df: float = 0.95,
    sublinear_tf: bool = True,
    use_class_weight_balanced: bool = True,
    random_state: int = 42,
) -> Pipeline:
    """
    TF-IDF + Linear SVM (LinearSVC).
    LinearSVC is typically the go-to for sparse high-dim text features.
    """
    clf = LinearSVC(
        class_weight="balanced" if use_class_weight_balanced else None,
        random_state=random_state,
    )

    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    strip_accents="unicode",
                    ngram_range=ngram_range,
                    min_df=min_df,
                    max_df=max_df,
                    sublinear_tf=sublinear_tf,
                ),
            ),
            ("svm", clf),
        ]
    )

In [ ]:
n_splits: int = 5
shuffle: bool = True
random_state: int = 42
verbose: bool = True

X = np.asarray(cleanedData, dtype=object)
y = np.asarray(data["label"])

if X.ndim != 1:
  raise ValueError("`texts` must be a 1D sequence of strings.")
if len(X) != len(y):
  raise ValueError(f"Mismatched lengths: len(texts)={len(X)} vs len(labels)={len(y)}")

skf = StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
pipeline = build_pipeline(random_state=random_state)


In [ ]:
fold_results: List[CVResult] = []
y_true_all=[]
y_pred_all=[]
classes = np.unique(y)

In [ ]:
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
  X_train, X_test = X[train_idx], X[test_idx]
  y_train, y_test = y[train_idx], y[test_idx]

  pipeline.fit(X_train, y_train)
  y_pred = pipeline.predict(X_test)
  # Aggregate metrics over all out-of-fold predictions
  y_true_all.extend(np.asarray(y_test))
  y_pred_all.extend(np.asarray(y_pred))

In [ ]:
len(y_true_all)

2157

In [ ]:
acc = accuracy_score(y_true_all, y_pred_all)
f1_mac = f1_score(y_true_all, y_pred_all, average="macro", zero_division=0)
f1_w = f1_score(y_true_all, y_pred_all, average="weighted", zero_division=0)

prec_mac, rec_mac, _, _ = precision_recall_fscore_support(y_true_all, y_pred_all, average="macro", zero_division=0)

cm = confusion_matrix(y_true_all, y_pred_all, labels=classes)

t_res= CVResult(
              accuracy=float(acc),
              f1_macro=float(f1_mac),
              f1_weighted=float(f1_w),
              precision_macro=float(prec_mac),
              recall_macro=float(rec_mac),
              support=int(len(test_idx)),
              confusion=cm,
          )


In [ ]:
  print(f"Accuracy     : {acc:.4f}")
  print(f"F1 (macro)   : {f1_mac:.4f}")
  print(f"F1 (weighted): {f1_w:.4f}")
  print("\nClassification report:")
  print(classification_report(y_true_all, y_pred_all, zero_division=0,  digits=3 ))

Accuracy     : 0.9254
F1 (macro)   : 0.6052
F1 (weighted): 0.9145

Classification report:
              precision    recall  f1-score   support

           0      0.636     0.357     0.458        98
           1      0.943     0.985     0.963      1961
           2      0.556     0.306     0.395        98

    accuracy                          0.925      2157
   macro avg      0.712     0.549     0.605      2157
weighted avg      0.911     0.925     0.915      2157

